# 01 — Data: Load & Preprocess

This notebook shows how to load the raw parquet files and run the preprocessing pipeline.

Output: `data/processed/dataset.parquet` — the cleaned, merged dataset ready for feature engineering.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.module.model.data import load_raw_data, preprocess
from utils.logger import get_logger

logger = get_logger('notebook')

## 1. Load Raw Data

`load_raw_data()` reads all year folders under `datasets/` and concatenates them into three DataFrames: matches, maps, player_stats.

In [ ]:
raw = load_raw_data()

for name, df in raw.items():
    print(f'{name:15s}: {df.shape[0]:>8,} rows  x  {df.shape[1]} cols')

In [ ]:
raw['matches'].head(3)

In [ ]:
raw['maps'].head(3)

In [ ]:
raw['player_stats'].head(3)

## 2. Preprocess

`preprocess()` runs 7 steps:
1. Clean matches (parse dates, drop draws/nulls)
2. Aggregate map stats (rounds won/lost, round diff)
3. Aggregate player stats (mean/max/std per team per match)
4. Merge all three tables on `match_id`
5. Drop high-NaN columns (threshold 50%)
6. Fill remaining NaN with column median
7. Derive target column `winner = score_team1 > score_team2`

Result is saved to `data/processed/dataset.parquet`.

In [ ]:
df = preprocess(raw)

print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Class balance (winner=1): {df["winner"].mean():.1%}')

In [ ]:
df.head(3)

## 3. Inspect Columns

In [ ]:
df.dtypes.to_frame('dtype')

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
null_pct[null_pct > 0].to_frame('null_pct').style.format('{:.1%}')

In [ ]:
df.describe().T

## 4. Load from Cache

If `dataset.parquet` already exists, skip preprocessing and load directly.

In [ ]:
from conf.settings import PROCESSED_DIR, DATASET_FILE

cached_path = PROCESSED_DIR / DATASET_FILE
df_cached = pd.read_parquet(cached_path)
print(f'Loaded from cache: {df_cached.shape}')
df_cached.head(2)